# FDI tooth detector — train on DENTEX (Colab)

Thin runner: this notebook only clones/pulls the repo and calls the scripts in `detector/`.
All the real code lives in the repo; edit + push there, then re-run — never paste code here.

**First:** Runtime → Change runtime type → **T4 GPU**.


In [ ]:
# 1. GPU + Google Drive (dataset & weights persist on Drive)
from google.colab import drive; drive.mount('/content/drive')
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > Change runtime type > T4 GPU')


In [ ]:
# 2. get the latest code + install deps (re-run this cell after any push)
import os
if not os.path.exists('dental'):
    !git clone -q https://github.com/leostuy2028/dental.git
!cd dental && git pull -q
!pip install -q -r dental/detector/requirements.txt
DATA = '/content/drive/MyDrive/dentex_yolo'   # dataset + runs live here


## Stage 0 — build the dataset (run ONCE; it writes to Drive)
Streams DENTEX set (b), cleans it, converts to YOLO format, splits, writes `dentex.yaml`.
Skip this cell on later sessions — the data is already on Drive.


In [ ]:
!python dental/detector/prepare_data.py --out {DATA}


## Stage 2 — train (≈1 hr on T4). Progress prints live; plots save to Drive.


In [ ]:
!python dental/detector/train.py --data {DATA}/dentex.yaml --project {DATA}/runs --epochs 120 --imgsz 1024 --batch 8


## Stage 3 — validate: mAP + count/FDI accuracy + worst errors (the gate)


In [ ]:
W = f'{DATA}/runs/dentex_yolov8n/weights/best.pt'
!python dental/detector/validate.py --weights {W} --data {DATA}/dentex.yaml


In [ ]:
# show the training curves + confusion matrix inline
from IPython.display import Image, display
R = f'{DATA}/runs/dentex_yolov8n'
for p in ('results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg'):
    if os.path.exists(f'{R}/{p}'): display(Image(f'{R}/{p}'))


## Stage 1 (optional) — SSL pretraining on unlabeled X-rays
Only if you want the technical-depth component. See `detector/README.md` for the
ResNet-vs-YOLO-backbone caveat. Extract the unlabeled images first, then:
```
!python dental/detector/pretrain_ssl.py --images-dir /content/unlabelled --out {DATA}/ssl_backbone.pt
```


## Stage 4 — run on MMOral
Runs **locally** in your `dental` repo (CPU), not here — download `best.pt` from Drive, then:
```
python detector/infer_mmoral.py --weights best.pt
```
